# Table 1: crisis overview

Columns C1 ... C7.  First row: duration of the crisis in S&P 500 trading days from the peak to the trough (crises.py).  Then the maximum drawdown within each crisis (S&P 500 peak -> back at the peak) for GSPC, MCap, 1/N, HRP, MVO, LSTM, RF, XGB (baselines, monthly).

Output: `data_finalized/table_1_crisis_overview.csv`

In [ ]:
'''
Table 1: crisis overview.  Duration per crisis (S&P 500 trading days peak ->
trough), then the maximum drawdown within each crisis (S&P 500 peak -> back
at the peak) per portfolio.
Portfolios: sheets C1 ... C7 of crisis_analysis_all_monthly.xlsx (column
full_max_dd, crisis_analysis_defined.py).  GSPC is not in that workbook
(sp500 is skipped there), so its row is computed with the same window_row().
Written for reading: days as whole numbers, drawdowns in % with one decimal.
'''

import sys
from pathlib import Path

import pandas as pd

HERE  = Path.cwd()                              # _latex, or the workspace root
PAPER = HERE.parent if (HERE.parent / '_comparison').is_dir() else HERE / 'Paper'
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics')]
import compare_all as ca
import crisis_analysis_defined as cad
from crises import main_crises

FREQ     = 'monthly'
OUT_PATH = PAPER / '_latex' / 'data_finalized' / 'table_1_crisis_overview.csv'

# name in the table -> output folder (_output/<model>/<folder>), in table order
GSPC = ('GSPC', 's&p_500')
RUNS = {
    'MCap': 'mcap_t_10_monthly',
    '1/N':  'equal_t_10_monthly',
    'HRP':  'hrp_t_10_lw_ward_monthly',
    'MVO':  'mvo_t_10_lw_monthly',
    'LSTM': 'lstm_t_10_holdout_rank_monthly',
    'RF':   'rf_t_10_holdout_monthly',
    'XGB':  'xgb_t_10_holdout_monthly',
}

crisis_ids = [f'C{i}' for i in range(1, len(main_crises) + 1)]
windows    = {name: w for name, w, phases in cad.WINDOWS if phases}             # C1 ... C7

sp = next(r for r in ca.find_runs() if r['run'] == GSPC[1] and r['frequency'] == FREQ)

# ---- duration: S&P 500 trading days in (peak, trough]
days = {c: len(ca._slice(sp['returns'], windows[c]['peak'], windows[c]['trough'])) for c in crisis_ids}

# ---- GSPC: sp500 run, same window_row() as the workbook
dd = {GSPC[0]: {c: cad.window_row(sp, sp['returns'], windows[c], phases=True)['full_max_dd']
                for c in crisis_ids}}

# ---- portfolios: full_max_dd from the crisis sheets
sheets = pd.read_excel(cad.SAVE_DIR / f'crisis_analysis_all_{FREQ}.xlsx', sheet_name=crisis_ids)
for label, folder in RUNS.items():
    run = folder.removesuffix(f'_{FREQ}')
    vals = {}
    for c in crisis_ids:
        hit = sheets[c][(sheets[c]['run'] == run) & (sheets[c]['frequency'] == FREQ)]
        if hit.empty:
            raise KeyError(f'{folder!r} not in the {FREQ} {c} sheet')
        vals[c] = hit.iloc[0]['full_max_dd']
    dd[label] = vals

# ---- readable cells: days as integers, drawdowns x 100 with one decimal and '%'
rows = [pd.Series({c: f'{days[c]:d}' for c in crisis_ids}, name='days')]
rows += [pd.Series({c: f'{100 * v:.1f}%' for c, v in vals.items()}, name=label)
         for label, vals in dd.items()]
table_1 = pd.DataFrame(rows)[crisis_ids].rename_axis('model').reset_index()

OUT_PATH.parent.mkdir(exist_ok=True)
table_1.to_csv(OUT_PATH, index=False)
print(f'wrote {OUT_PATH}: {table_1.shape[0]} rows x {table_1.shape[1] - 1} columns')
table_1

# Table 2: full-period performance

Rows as in Table 1 (GSPC, MCap, 1/N, HRP, MVO, LSTM, RF, XGB; baselines, monthly).  Columns over the full period 1998-2025:

|sharpe_ratio|cagr|cum_return|annual_volatility|max_drawdown|ulcer_index|turnover_pa|

Output: `data_finalized/table_2_full_period.csv`

In [ ]:
'''
Table 2: full-period performance 1998-2025 (crises.overall).
Portfolios: overall sheet of crisis_analysis_all_monthly.xlsx
(crisis_analysis_defined.py).  GSPC is not in that workbook (sp500 is
skipped there), so its row is computed with the same window_row().
Written for reading, decimals per column (COLUMNS).  GSPC is buy & hold:
no turnover after the initial purchase (0%).
'''

import sys
from pathlib import Path

import pandas as pd

HERE  = Path.cwd()                              # _latex, or the workspace root
PAPER = HERE.parent if (HERE.parent / '_comparison').is_dir() else HERE / 'Paper'
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics')]
import compare_all as ca
import crisis_analysis_defined as cad
from crises import overall

FREQ       = 'monthly'
OUT_PATH_2 = PAPER / '_latex' / 'data_finalized' / 'table_2_full_period.csv'

# name in the table -> output folder (_output/<model>/<folder>), in table order
GSPC = ('GSPC', 's&p_500')
RUNS = {
    'MCap': 'mcap_t_10_monthly',
    '1/N':  'equal_t_10_monthly',
    'HRP':  'hrp_t_10_lw_ward_monthly',
    'MVO':  'mvo_t_10_lw_monthly',
    'LSTM': 'lstm_t_10_holdout_rank_monthly',
    'RF':   'rf_t_10_holdout_monthly',
    'XGB':  'xgb_t_10_holdout_monthly',
}

# table column -> (source column, format); '{:.1%}' = x 100 with one decimal and '%'
COLUMNS = {
    'sharpe_ratio':      ('full_sharpe',      '{:.2f}'),
    'cagr':              ('full_cagr',        '{:.2%}'),
    'cum_return':        ('full_cum_return',  '{:.0%}'),
    'annual_volatility': ('full_ann_vol',     '{:.1%}'),
    'max_drawdown':      ('full_max_dd',      '{:.1%}'),
    'ulcer_index':       ('full_ulcer',       '{:.1%}'),
    'turnover_pa':       ('full_turnover_pa', '{:.0%}'),
}

# ---- GSPC: sp500 run, same window_row() as the workbook's overall sheet
sp = next(r for r in ca.find_runs() if r['run'] == GSPC[1] and r['frequency'] == FREQ)
values = {GSPC[0]: cad.window_row(sp, sp['returns'], overall[0], phases=False)}
values[GSPC[0]]['full_turnover_pa'] = 0.0               # buy & hold

# ---- portfolios: overall sheet
src = pd.read_excel(cad.SAVE_DIR / f'crisis_analysis_all_{FREQ}.xlsx', sheet_name='overall')
src = src[src['frequency'] == FREQ].set_index('run')
for label, folder in RUNS.items():
    run = folder.removesuffix(f'_{FREQ}')
    if run not in src.index:
        raise KeyError(f'{folder!r} not in the {FREQ} overall sheet')
    values[label] = src.loc[run]

table_2 = pd.DataFrame(
    [pd.Series({col: f.format(values[label][s]) for col, (s, f) in COLUMNS.items()}, name=label)
     for label in [GSPC[0], *RUNS]]).rename_axis('model').reset_index()

OUT_PATH_2.parent.mkdir(exist_ok=True)
table_2.to_csv(OUT_PATH_2, index=False)
print(f'wrote {OUT_PATH_2}: {table_2.shape[0]} rows x {table_2.shape[1] - 1} columns')
table_2

# Table 3: average crisis performance

Rows as in Table 1.  Each column is the equal-weighted average over the seven crises C1 ... C7:

|avg_max_dd|avg_ulcer|avg_recovery_days|avg_return_full|avg_return_ptt|avg_return_ttp|

Fixed S&P 500 windows: full = peak -> back at the peak, ptt = peak -> trough, ttp = trough -> back at the peak.  Max. DD and Ulcer index over the full window; cumulative return over each window.  Recovery: trading days from the portfolio's own trough until it is back at its value on the S&P peak date, searched up to the next crisis peak (C7: end of the data); a portfolio not recovered by then counts up to that cut-off, and its average is marked as a lower bound (≥).

Output: `data_finalized/table_3_crisis_average.csv`

In [ ]:
'''
Table 3: average crisis performance, equal weight over C1 ... C7.
Computed from each run's daily returns with the functions behind the crisis
workbooks, so GSPC and the portfolios are treated the same:
  max. DD, Ulcer       crisis_analysis_defined.full_metrics, fixed window
                       (S&P peak, S&P back at the peak]
  recovery             crisis_analysis_variable.episode: ttp_days = trading
                       days own trough -> back at the anchor (value on the S&P
                       peak date), searched up to the next crisis peak
                       (C7: end of the data); full_not_recovered = cut off
  cumulative return    metrics.cumulative_return over (peak, even],
                       (peak, trough], (trough, even]
Written for reading; an average that includes a cut-off recovery is a lower
bound and starts with '≥'.
'''

import sys
from pathlib import Path

import numpy as np
import pandas as pd

HERE  = Path.cwd()                              # _latex, or the workspace root
PAPER = HERE.parent if (HERE.parent / '_comparison').is_dir() else HERE / 'Paper'
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics')]
import compare_all as ca
import crisis_analysis_defined as cad
import crisis_analysis_variable as cav
from crises import main_crises
from metrics import cumulative_return

FREQ       = 'monthly'
OUT_PATH_3 = PAPER / '_latex' / 'data_finalized' / 'table_3_crisis_average.csv'

# name in the table -> output folder (_output/<model>/<folder>), in table order
RUNS = {
    'GSPC': 's&p_500',
    'MCap': 'mcap_t_10_monthly',
    '1/N':  'equal_t_10_monthly',
    'HRP':  'hrp_t_10_lw_ward_monthly',
    'MVO':  'mvo_t_10_lw_monthly',
    'LSTM': 'lstm_t_10_holdout_rank_monthly',
    'RF':   'rf_t_10_holdout_monthly',
    'XGB':  'xgb_t_10_holdout_monthly',
}

by_run = {r['run']: r for r in ca.find_runs() if r['frequency'] == FREQ}

def crisis_values(ret: pd.Series, i: int, c: dict) -> dict:
    '''All Table 3 metrics of one run in crisis c (index i in main_crises).'''
    s = lambda a, b: ca._slice(ret, c[a], c[b])
    full = cad.full_metrics(s('peak', 'even'), pd.Series(dtype=float))
    rec_end = main_crises[i + 1]['peak'] if i + 1 < len(main_crises) else None
    ep = cav.episode(ret, c, rec_end)
    return {
        'max_dd':        full['full_max_dd'],
        'ulcer':         full['full_ulcer'],
        'recovery_days': ep['ttp_days'],
        'censored':      ep['full_not_recovered'],
        'return_full':   cumulative_return(s('peak', 'even')),
        'return_ptt':    cumulative_return(s('peak', 'trough')),
        'return_ttp':    cumulative_return(s('trough', 'even')),
    }

rows, censored = [], {}
for label, folder in RUNS.items():
    run = folder.removesuffix(f'_{FREQ}')
    if run not in by_run:
        raise KeyError(f'no {FREQ} run {folder!r}')
    per = pd.DataFrame([crisis_values(by_run[run]['returns'], i, c) for i, c in enumerate(main_crises)],
                       index=[f'C{i}' for i in range(1, len(main_crises) + 1)])
    censored[label] = per.index[per['censored']].tolist()
    avg = per.drop(columns='censored').mean()
    low = '≥' if censored[label] else ''
    rows.append(pd.Series({
        'avg_max_dd':        f'{avg["max_dd"]:.1%}',
        'avg_ulcer':         f'{avg["ulcer"]:.1%}',
        'avg_recovery_days': f'{low}{avg["recovery_days"]:.0f}',
        'avg_return_full':   f'{avg["return_full"]:.1%}',
        'avg_return_ptt':    f'{avg["return_ptt"]:.1%}',
        'avg_return_ttp':    f'{avg["return_ttp"]:.1%}',
    }, name=label))

table_3 = pd.DataFrame(rows).rename_axis('model').reset_index()

OUT_PATH_3.parent.mkdir(exist_ok=True)
table_3.to_csv(OUT_PATH_3, index=False, encoding='utf-8')
print(f'wrote {OUT_PATH_3}: {table_3.shape[0]} rows x {table_3.shape[1] - 1} columns')
print('recovery cut off at the next crisis peak:', {k: v for k, v in censored.items() if v})
table_3

# Table 4: regime detectors

12 rows: changepoint (VIX), HMM (S&P 500), clustering (Wasserstein k-means, S&P 500 + VIX), grouped by daily / monthly / quarterly / yearly.  Daily = the detector's own daily state; monthly / quarterly / yearly = the state on the first trading day of the month / quarter / year, held until the next one (as the portfolios hold it).

|n_episodes|avg_episode_days|defensive_share|avg_entry_lag|avg_exit_lag|recall|precision|

Episodes: number of switches into the crisis state and their average length (trading days).  Entry lag: trading days from the S&P peak to the first crisis day (negative = already in crisis at the peak; averaged over the crises entered before the trough).  Exit lag: trading days from the S&P trough to the first calm day (averaged over the crises in which the detector switches back).  Number of crises behind each average lag in brackets.  Recall / precision per day against the label S&P peak -> trough.

Output: `data_finalized/table_4_detectors.csv`

In [ ]:
'''
Table 4: regime detectors, independent of the portfolio models.
States and metrics from detector_evaluation.py, recomputed here:
  daily      detector_evaluation.daily_state: crisis on t if the stored
             signal of t-1 is above regime_def.CRISIS_THRESHOLD
  monthly / quarterly / yearly
             detector_evaluation.applied_states: the crisis flag of the regime
             runs (regime.csv), set on the first trading day of the month /
             quarter / year and held from the next day until the next one
  per crisis crisis_metrics: exit_lag (S&P peak -> first crisis day; here the
             entry lag), reentry_lag (S&P trough -> first calm day; here the
             exit lag); NaN where the detector never switches
  full period  full_period: n_episodes, mean_episode_days,
             total_defensive_share, recall_ptt, precision_ptt
Calendar: the trading days 1998-01-02 .. 2025-12-31 of the portfolio runs.
The Wasserstein k-means detector is called "Clustering" in the paper.
'''

import sys
from pathlib import Path

import numpy as np
import pandas as pd

HERE  = Path.cwd()                              # _latex, or the workspace root
PAPER = HERE.parent if (HERE.parent / '_comparison').is_dir() else HERE / 'Paper'
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics')]
import compare_all as ca
import crisis_analysis_defined as cad
import detector_evaluation as de
from crises import overall, main_crises

OUT_PATH_4  = PAPER / '_latex' / 'data_finalized' / 'table_4_detectors.csv'
FREQUENCIES = ['daily', 'monthly', 'quarterly', 'yearly']
# (detector, series) -> name in the table, in table order
DETECTORS = {('changepoint', 'vix'): 'Changepoint',
             ('hmm', 'gspc'): 'HMM',
             ('wasserstein', 'gspc_vix'): 'Clustering'}

runs     = ca.find_runs()
bench    = next(r for r in runs if r['model'] == cad.BENCHMARK and r['frequency'] == 'monthly')
calendar = ca._slice(bench['returns'], overall[0]['peak'], overall[0]['even']).index
sp       = next(r for r in runs if r['model'] == de.SP500_MODEL)['returns']
applied  = de.applied_states(runs)

def lag(values: list) -> str:
    '''Average over the crises with a value, and how many that are.'''
    v = np.array(values, dtype=float)
    n = int(np.isfinite(v).sum())
    return f'{np.nanmean(v):.1f} ({n})' if n else f'-- (0)'

rows = []
for freq in FREQUENCIES:
    for (det, ser), name in DETECTORS.items():
        if freq == 'daily':
            d = de.daily_state(det, ser, calendar)
        else:
            if (det, ser, freq) not in applied:
                raise KeyError(f'no {freq} regime run for {det} {ser}')
            d = applied[(det, ser, freq)].reindex(calendar, fill_value=False)
        pc = [de.crisis_metrics(d, k) for k in range(len(main_crises))]
        full, _ = de.full_period(d, pc, sp, {})
        rows.append({
            'frequency':        freq,
            'detector':         name,
            'n_episodes':       f'{full["n_episodes"]:d}',
            'avg_episode_days': f'{full["mean_episode_days"]:.0f}',
            'defensive_share':  f'{full["total_defensive_share"]:.1%}',
            'avg_entry_lag':    lag([m['exit_lag'] for m in pc]),
            'avg_exit_lag':     lag([m['reentry_lag'] for m in pc]),
            'recall':           f'{full["recall_ptt"]:.1%}',
            'precision':        f'{full["precision_ptt"]:.1%}',
        })

table_4 = pd.DataFrame(rows)

OUT_PATH_4.parent.mkdir(exist_ok=True)
table_4.to_csv(OUT_PATH_4, index=False, encoding='utf-8')
print(f'wrote {OUT_PATH_4}: {table_4.shape[0]} rows x {table_4.shape[1] - 2} columns')
table_4

# Figure: monthly crisis probability of the detectors

One row per month 1998-01 ... 2025-12: the first trading day of the month and the crisis probability $p_{crisis}$ each detector gives on that day (the value the monthly regime runs read on their rebalancing date; it uses data through the day before).  Columns in chart order, top to bottom:

|date|changepoint|clustering|hmm|

Changepoint = VIX, clustering = Wasserstein k-means on S&P 500 + VIX, HMM = S&P 500.  Checked: $p_{crisis} > 0.5$ equals the crisis flag of the monthly regime runs on every date.

Output: `data_finalized/figure_pcrisis_monthly.csv`

In [ ]:
'''
Figure: monthly crisis probability of the three detectors, for the stacked
line chart in paper.tex (top to bottom: changepoint, clustering, HMM).
Dates: first trading day of each month in the calendar of the portfolio runs
(1998-01-02 .. 2025-12-31), i.e. the monthly rebalancing dates.
Value: regime_def._signal, forward-filled to that day, exactly as
regime_def.regime_probs reads it on a rebalancing date (stored with a one-day
shift, so it uses data through the day before).
Check: p_crisis > CRISIS_THRESHOLD equals the crisis flag (regime.csv) of the
monthly regime runs on every date.
'''

import sys
from pathlib import Path

import pandas as pd

HERE  = Path.cwd()                              # _latex, or the workspace root
PAPER = HERE.parent if (HERE.parent / '_comparison').is_dir() else HERE / 'Paper'
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics'), str(PAPER / '_regimes')]
import compare_all as ca
import crisis_analysis_defined as cad
import regime_def
from crises import overall

OUT_PATH_F = PAPER / '_latex' / 'data_finalized' / 'figure_pcrisis_monthly.csv'
FREQ = 'monthly'
# column -> (detector, series), in chart order (top to bottom)
DETECTORS = {'changepoint': ('changepoint', 'vix'),
             'clustering':  ('wasserstein', 'gspc_vix'),
             'hmm':         ('hmm', 'gspc')}

runs     = ca.find_runs()
bench    = next(r for r in runs if r['model'] == cad.BENCHMARK and r['frequency'] == FREQ)
calendar = ca._slice(bench['returns'], overall[0]['peak'], overall[0]['even']).index
dates    = calendar.to_series().groupby(calendar.to_period('M')).first()

figure = pd.DataFrame({'date': dates.dt.strftime('%Y-%m-%d').values})
for col, (det, ser) in DETECTORS.items():
    sig = regime_def._signal(det, ser, regime_def.SMOOTH_HALFLIFE)['p_crisis']
    p = sig.reindex(dates.values, method='ffill')
    if p.isna().any():
        raise ValueError(f'{col}: no signal on {p.index[p.isna()][0].date()}')
    run = next((r for r in runs if (r['detector'], r['series'], r['frequency']) == (det, ser, FREQ)), None)
    if run is None:
        raise KeyError(f'no {FREQ} regime run for {det} {ser}')
    flag = run['crisis'].reindex(dates.values, method='ffill') > regime_def.CRISIS_THRESHOLD
    if not ((p > regime_def.CRISIS_THRESHOLD) == flag).all():
        raise AssertionError(f'{col}: p_crisis does not match the {FREQ} regime run on '
                             f'{((p > regime_def.CRISIS_THRESHOLD) != flag).sum()} dates')
    figure[col] = p.round(4).values

OUT_PATH_F.parent.mkdir(exist_ok=True)
figure.to_csv(OUT_PATH_F, index=False)
print(f'wrote {OUT_PATH_F}: {len(figure)} months, {figure["date"].iloc[0]} .. {figure["date"].iloc[-1]}')
print('months in crisis (p > 0.5):', (figure[list(DETECTORS)] > regime_def.CRISIS_THRESHOLD).sum().to_dict())
figure

# Table 6: detector influence

7 rows: baselines, then per detector (changepoint, clustering, HMM) the cash and the alternative variant.  Each row is one number per column, averaged over the five models HRP, MVO, LSTM, RF, XGB (monthly rebalancing).  Cash: the crisis portfolio is T-bills.  Alternative: MVO and HRP switch to minimum variance, LSTM, RF and XGB switch to HRP.

|ret_ptt|ret_ttp|ret_full|max_dd|ulcer|sharpe_crisis|sharpe_outside|

Aggregation: per model, the mean over the seven crises C1 ... C7 (equal weight); then the mean over the five models (equal weight).  Fixed S&P 500 windows: ptt = peak -> trough, ttp = trough -> back at the peak, full = peak -> back at the peak; Max. DD and Ulcer index over the full window.  Sharpe ratios pooled, one per model: in crisis over the daily returns of all seven full windows together, outside crisis over all other trading days 1998-2025; then the mean over the five models.  Sharpe ratios annualised, in excess of the T-bill rate.

Output: `data_finalized/table_6_detector_influence.csv`

In [ ]:
'''
Table 6: detector influence.  Two-step equal-weight average:
  1. per model: mean over C1 ... C7 of the crisis value (Sharpe ratios: pooled)
  2. per row:   mean over HRP, MVO, LSTM, RF, XGB
Computed from each run's daily returns with the functions behind Table 3:
  cumulative return    metrics.cumulative_return over (peak, trough],
                       (trough, even], (peak, even]
  max. DD, Ulcer       crisis_analysis_defined.full_metrics over (peak, even]
  Sharpe ratios        metrics.sharpe_ratio (excess of the T-bill rate,
                       annualised), pooled, one per model: in crisis over the
                       daily returns of all seven (peak, even] windows
                       together, outside over the remaining trading days of
                       crises.overall; so no step 1 for these two columns
Run folders: <baseline>_<detector>_<series>_<variant>_monthly; variant cash,
or the alternative crisis portfolio (min_var for HRP and MVO, hrp for LSTM,
RF and XGB).
'''

import sys
from pathlib import Path

import pandas as pd

HERE  = Path.cwd()                              # _latex, or the workspace root
PAPER = HERE.parent if (HERE.parent / '_comparison').is_dir() else HERE / 'Paper'
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics')]
import compare_all as ca
import crisis_analysis_defined as cad
from crises import overall, main_crises
from metrics import cumulative_return, sharpe_ratio

FREQ       = 'monthly'
OUT_PATH_6 = PAPER / '_latex' / 'data_finalized' / 'table_6_detector_influence.csv'

# model -> (baseline run, alternative crisis portfolio)
MODELS = {
    'HRP':  ('hrp_t_10_lw_ward',       'min_var'),
    'MVO':  ('mvo_t_10_lw',            'min_var'),
    'LSTM': ('lstm_t_10_holdout_rank', 'hrp'),
    'RF':   ('rf_t_10_holdout',        'hrp'),
    'XGB':  ('xgb_t_10_holdout',       'hrp'),
}
# row name -> (detector_series, variant); variant 'alt' = the model's alternative
ROWS = {
    'Baseline':                (None, None),
    'Changepoint Cash':        ('changepoint_vix', 'cash'),
    'Changepoint Alternative': ('changepoint_vix', 'alt'),
    'Clustering Cash':         ('wasserstein_gspc_vix', 'cash'),
    'Clustering Alternative':  ('wasserstein_gspc_vix', 'alt'),
    'HMM Cash':                ('hmm_gspc', 'cash'),
    'HMM Alternative':         ('hmm_gspc', 'alt'),
}
# table column -> format; '{:.1%}' = x 100 with one decimal and '%'
COLUMNS = {
    'ret_ptt':        '{:.1%}',
    'ret_ttp':        '{:.1%}',
    'ret_full':       '{:.1%}',
    'max_dd':         '{:.1%}',
    'ulcer':          '{:.1%}',
    'sharpe_crisis':  '{:.2f}',
    'sharpe_outside': '{:.2f}',
}

by_run = {r['run']: r for r in ca.find_runs() if r['frequency'] == FREQ}

def run_name(base: str, alt: str, det: str | None, variant: str | None) -> str:
    if det is None:
        return base
    return f'{base}_{det}_{alt if variant == "alt" else variant}'

def split(ret: pd.Series) -> tuple[pd.Series, pd.Series]:
    '''Returns of crises.overall on the days of the seven (peak, even] windows, and on all other days.'''
    r = ca._slice(ret, overall[0]['peak'], overall[0]['even'])
    inside = pd.Index([])
    for c in main_crises:
        inside = inside.union(ca._slice(r, c['peak'], c['even']).index)
    return r.loc[inside], r.drop(inside)

def model_values(ret: pd.Series) -> dict:
    '''Step 1: crisis means over C1 ... C7, and the pooled Sharpe ratios in and outside crisis.'''
    per = []
    for c in main_crises:
        s = lambda a, b: ca._slice(ret, c[a], c[b])
        full = cad.full_metrics(s('peak', 'even'), pd.Series(dtype=float))
        per.append({
            'ret_ptt':       cumulative_return(s('peak', 'trough')),
            'ret_ttp':       cumulative_return(s('trough', 'even')),
            'ret_full':      cumulative_return(s('peak', 'even')),
            'max_dd':        full['full_max_dd'],
            'ulcer':         full['full_ulcer'],
        })
    inside, outside = split(ret)
    return pd.DataFrame(per).mean().to_dict() | {'sharpe_crisis':  sharpe_ratio(inside),
                                                 'sharpe_outside': sharpe_ratio(outside)}

def cell(v: float, f: str) -> str:
    '''Formatted value; no sign on a value that rounds to zero ('-0.00' -> '0.00').'''
    s = f.format(v)
    return s[1:] if s.startswith('-') and not any(ch in s for ch in '123456789') else s

per_model, rows = {}, []
for label, (det, variant) in ROWS.items():
    vals = {}
    for model, (base, alt) in MODELS.items():
        run = run_name(base, alt, det, variant)
        if run not in by_run:
            raise KeyError(f'no {FREQ} run {run!r}')
        vals[model] = model_values(by_run[run]['returns'])
    per_model[label] = pd.DataFrame(vals).T[list(COLUMNS)]
    avg = per_model[label].mean()                                   # step 2
    rows.append(pd.Series({col: cell(avg[col], f) for col, f in COLUMNS.items()}, name=label))

table_6 = pd.DataFrame(rows).rename_axis('strategy').reset_index()

OUT_PATH_6.parent.mkdir(exist_ok=True)
table_6.to_csv(OUT_PATH_6, index=False, encoding='utf-8')
print(f'wrote {OUT_PATH_6}: {table_6.shape[0]} rows x {table_6.shape[1] - 1} columns')
# step 1 per model, for checking (not written)
display(pd.concat(per_model).round(3))
table_6

# Figure: portfolio value of HRP and LSTM with and without regime detection

One row per trading day 1998-2025, the value of 1 invested on the first day (log returns compounded), for the seven monthly portfolios of HRP and of LSTM: baseline, and per detector (changepoint, clustering, HMM) the cash and the alternative variant (HRP: minimum variance, LSTM: HRP).  Columns `<model>_<portfolio>`:

|date|hrp_baseline|hrp_changepoint_cash|hrp_changepoint_alt|hrp_clustering_cash|hrp_clustering_alt|hrp_hmm_cash|hrp_hmm_alt|lstm_baseline|...|lstm_hmm_alt|

Output: `data_finalized/figure_value_hrp_lstm.csv`

In [ ]:
'''
Figure: portfolio value of HRP and LSTM, seven monthly portfolios each, for
the stacked chart in paper.tex (top HRP, bottom LSTM).
Value: exp(cumulative sum of the daily log returns) over crises.overall,
1.0 on its first date (the day before the first return), as
compare_all._level.  All 14 runs share the trading calendar; a missing day
raises.
'''

import sys
from pathlib import Path

import numpy as np
import pandas as pd

HERE  = Path.cwd()                              # _latex, or the workspace root
PAPER = HERE.parent if (HERE.parent / '_comparison').is_dir() else HERE / 'Paper'
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics')]
import compare_all as ca
from crises import overall

FREQ       = 'monthly'
OUT_PATH_V = PAPER / '_latex' / 'data_finalized' / 'figure_value_hrp_lstm.csv'

# model -> (baseline run, alternative crisis portfolio)
MODELS = {
    'hrp':  ('hrp_t_10_lw_ward',       'min_var'),
    'lstm': ('lstm_t_10_holdout_rank', 'hrp'),
}
# portfolio -> run suffix ('alt' = the model's alternative), in legend order
PORTFOLIOS = {
    'baseline':         '',
    'changepoint_cash': '_changepoint_vix_cash',
    'changepoint_alt':  '_changepoint_vix_alt',
    'clustering_cash':  '_wasserstein_gspc_vix_cash',
    'clustering_alt':   '_wasserstein_gspc_vix_alt',
    'hmm_cash':         '_hmm_gspc_cash',
    'hmm_alt':          '_hmm_gspc_alt',
}

by_run = {r['run']: r for r in ca.find_runs() if r['frequency'] == FREQ}
start, end = overall[0]['peak'], overall[0]['even']

values = {}
for model, (base, alt) in MODELS.items():
    for name, suffix in PORTFOLIOS.items():
        run = base + suffix.replace('_alt', f'_{alt}')
        if run not in by_run:
            raise KeyError(f'no {FREQ} run {run!r}')
        r = ca._slice(by_run[run]['returns'], start, end)
        values[f'{model}_{name}'] = pd.concat([pd.Series([0.0], index=[pd.Timestamp(start)]), r]).cumsum().pipe(np.exp)

figure_v = pd.DataFrame(values)
if figure_v.isna().any().any():
    raise ValueError('runs do not share the trading calendar')

OUT_PATH_V.parent.mkdir(exist_ok=True)
figure_v.round(4).rename_axis('date').to_csv(OUT_PATH_V, date_format='%Y-%m-%d')
print(f'wrote {OUT_PATH_V}: {len(figure_v)} days, {figure_v.index[0].date()} .. {figure_v.index[-1].date()}')
print('range:', round(figure_v.min().min(), 2), '..', round(figure_v.max().max(), 2))
figure_v.iloc[-1].round(2)

# Figure: LSTM portfolios within each crisis

One file per crisis C1 ... C7, in full calendar months from the 1st of the month of the S&P 500 peak to the last day of the month of the break-even: the cumulative log return relative to the S&P peak (0 on the peak date) of MCap and of the seven monthly LSTM portfolios (baseline, and per detector the cash and the alternative (HRP) variant).  Weekly points (last trading day of each week) plus the exact peak, trough and break-even days.

|date|mcap|lstm_baseline|lstm_changepoint_cash|lstm_changepoint_alt|lstm_clustering_cash|lstm_clustering_alt|lstm_hmm_cash|lstm_hmm_alt|

Output: `data_finalized/figure_crises_lstm_c1.csv` ... `figure_crises_lstm_c7.csv`

In [ ]:
'''
Figure: LSTM portfolios within each crisis, for the 4 x 2 panel chart in
paper.tex (C1 ... C7, legend in the eighth slot).
Window: full calendar months, from the 1st of the month of the S&P peak to
the last day of the month of the S&P break-even (crises.main_crises).
Value: cumulative log return relative to the S&P peak date (0 on the peak;
before the peak minus the returns still to come until the peak).
Points: weekly, the last trading day of each week, plus the exact peak,
trough and break-even days and the first and last day of the window, so the
turning points are not smoothed away.  The values are cumulative, so picking
days needs no recalculation.
One file per crisis (figure_crises_lstm_c<k>.csv), so each panel reads only
its own rows and LaTeX has nothing to filter.
'''

import sys
from pathlib import Path

import pandas as pd

HERE  = Path.cwd()                              # _latex, or the workspace root
PAPER = HERE.parent if (HERE.parent / '_comparison').is_dir() else HERE / 'Paper'
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics')]
import compare_all as ca
from crises import main_crises

FREQ      = 'monthly'
OUT_DIR_C = PAPER / '_latex' / 'data_finalized'           # figure_crises_lstm_c<k>.csv

# column -> run, in plot order
RUNS = {
    'mcap':                  'mcap_t_10',
    'lstm_baseline':         'lstm_t_10_holdout_rank',
    'lstm_changepoint_cash': 'lstm_t_10_holdout_rank_changepoint_vix_cash',
    'lstm_changepoint_alt':  'lstm_t_10_holdout_rank_changepoint_vix_hrp',
    'lstm_clustering_cash':  'lstm_t_10_holdout_rank_wasserstein_gspc_vix_cash',
    'lstm_clustering_alt':   'lstm_t_10_holdout_rank_wasserstein_gspc_vix_hrp',
    'lstm_hmm_cash':         'lstm_t_10_holdout_rank_hmm_gspc_cash',
    'lstm_hmm_alt':          'lstm_t_10_holdout_rank_hmm_gspc_hrp',
}

by_run = {r['run']: r for r in ca.find_runs() if r['frequency'] == FREQ}
missing = [run for run in RUNS.values() if run not in by_run]
if missing:
    raise KeyError(f'no {FREQ} runs {missing}')

OUT_DIR_C.mkdir(exist_ok=True)
summary = {}
for k, c in enumerate(main_crises, 1):
    peak, trough, even = (pd.Timestamp(c[x]) for x in ('peak', 'trough', 'even'))
    start, end = peak.replace(day=1), even + pd.offsets.MonthEnd(0)
    cols = {}
    for col, run in RUNS.items():
        cum = by_run[run]['returns'].loc[start:end].cumsum()
        if peak not in cum.index:
            raise KeyError(f'C{k}: {run} has no return on the peak {peak.date()}')
        cols[col] = cum - cum.loc[peak]                     # 0 on the peak
    part = pd.DataFrame(cols)
    if part.isna().any().any():
        raise ValueError(f'C{k}: runs do not share the trading calendar')
    days = part.index.to_series()
    keep = pd.DatetimeIndex(days.groupby(days.index.to_period('W-FRI')).last())
    keep = keep.union([days.index[0], days.index[-1], peak, trough, even])
    part = part.loc[keep]
    out = OUT_DIR_C / f'figure_crises_lstm_c{k}.csv'
    part.round(4).rename_axis('date').to_csv(out, date_format='%Y-%m-%d')
    print(f'wrote {out.name}: {len(part)} rows, {part.index[0].date()} .. {part.index[-1].date()} '
          f'(window {start.date()} .. {end.date()})')
    summary[f'C{k}'] = part.loc[even]

pd.DataFrame(summary).round(3)          # value on the break-even day

# Figure: crisis excess return by rebalancing frequency

19 points: MCap (monthly, the benchmark, at 0 / 0) and 1/N, HRP, MVO, LSTM, RF, XGB (baselines) at monthly, quarterly and yearly rebalancing.  Per point the average over C1 ... C7 (equal weight) of the excess return over MCap monthly, peak -> trough (x) and trough -> back at the peak (y), fixed S&P 500 windows.  Excess return as in crisis_analysis_defined: (1 + R) / (1 + R_MCap) - 1, in %.  One row per model, the three frequencies side by side, so the arrows monthly -> quarterly -> yearly come straight from the columns (MCap: quarterly and yearly nan).

|model|key|ptt_monthly|ttp_monthly|ptt_quarterly|ttp_quarterly|ptt_yearly|ttp_yearly|label_pos|

`key`: colour class in paper.tex; `label_pos`: side of the name label at the monthly point.

Output: `data_finalized/figure_scatter_frequency.csv`

In [ ]:
'''
Figure: crisis excess return over MCap monthly by rebalancing frequency, for
the scatter plot in paper.tex (19 points, arrows monthly -> quarterly -> yearly).
Per run and crisis: cumulative return (metrics.cumulative_return) over the
fixed windows (peak, trough] and (trough, even], excess over MCap monthly in
the same window as crisis_analysis_defined._excess, (1 + R) / (1 + R_b) - 1;
then the mean over C1 ... C7.  Values in %.
'''

import sys
from pathlib import Path

import numpy as np
import pandas as pd

HERE  = Path.cwd()                              # _latex, or the workspace root
PAPER = HERE.parent if (HERE.parent / '_comparison').is_dir() else HERE / 'Paper'
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics')]
import compare_all as ca
import crisis_analysis_defined as cad
from crises import main_crises
from metrics import cumulative_return

OUT_PATH_S  = PAPER / '_latex' / 'data_finalized' / 'figure_scatter_frequency.csv'
FREQUENCIES = ['monthly', 'quarterly', 'yearly']
BENCHMARK   = ('mcap_t_10', 'monthly')
# name in the figure -> baseline run, in table order
MODELS = {
    '1/N':  'equal_t_10',
    'HRP':  'hrp_t_10_lw_ward',
    'MVO':  'mvo_t_10_lw',
    'LSTM': 'lstm_t_10_holdout_rank',
    'RF':   'rf_t_10_holdout',
    'XGB':  'xgb_t_10_holdout',
}
# name -> (key for the colour class in paper.tex, side of the name label at
# the monthly point: north / south / east / west), chosen for the layout
LABELS = {
    'MCap': ('mcap', 'west'),
    '1/N':  ('equal', 'south'),
    'HRP':  ('hrp', 'north'),
    'MVO':  ('mvo', 'west'),
    'LSTM': ('lstm', 'north'),
    'RF':   ('rf', 'west'),
    'XGB':  ('xgb', 'south'),
}

by_key = {(r['run'], r['frequency']): r for r in ca.find_runs()}

def phase_returns(key: tuple) -> pd.DataFrame:
    '''Cumulative return per crisis (rows C1 ... C7), columns ptt / ttp.'''
    if key not in by_key:
        raise KeyError(f'no run {key}')
    ret = by_key[key]['returns']
    return pd.DataFrame([{'ptt': cumulative_return(ca._slice(ret, c['peak'], c['trough'])),
                          'ttp': cumulative_return(ca._slice(ret, c['trough'], c['even']))}
                         for c in main_crises])

bench = phase_returns(BENCHMARK)

def excess(key: tuple) -> pd.Series:
    '''Mean over C1 ... C7 of the excess return over MCap monthly, in %.'''
    r = phase_returns(key)
    return 100 * ((1 + r) / (1 + bench) - 1).mean()

rows = [{'model': 'MCap', 'ptt_monthly': 0.0, 'ttp_monthly': 0.0}
        | {f'{p}_{f}': np.nan for f in FREQUENCIES[1:] for p in ('ptt', 'ttp')}]
for name, run in MODELS.items():
    row = {'model': name}
    for f in FREQUENCIES:
        e = excess((run, f))
        row[f'ptt_{f}'], row[f'ttp_{f}'] = e['ptt'], e['ttp']
    rows.append(row)

figure_s = pd.DataFrame(rows)
figure_s.insert(1, 'key', figure_s['model'].map(lambda m: LABELS[m][0]))
figure_s['label_pos'] = figure_s['model'].map(lambda m: LABELS[m][1])

OUT_PATH_S.parent.mkdir(exist_ok=True)
figure_s.round(2).to_csv(OUT_PATH_S, index=False, na_rep='nan')
print(f'wrote {OUT_PATH_S}: {len(figure_s)} rows, '
      f'{1 + (len(figure_s) - 1) * len(FREQUENCIES)} points')
figure_s.round(2)

# Figure: crisis excess return of the LSTM regime portfolios

8 points: MCap (monthly, the benchmark, at 0 / 0) and the seven monthly LSTM portfolios: baseline, and per detector (changepoint, clustering, HMM) the cash and the alternative (HRP) variant.  Same measure as the frequency scatter: per point the average over C1 ... C7 of the excess return over MCap monthly, (1 + R) / (1 + R_MCap) - 1 in %, peak -> trough (x) and trough -> back at the peak (y).  Arrows from the LSTM baseline to each variant: `dx`, `dy` (variant minus baseline; nan for MCap and the baseline).

|model|key|ptt|ttp|dx|dy|label_pos|

Output: `data_finalized/figure_scatter_lstm_regimes.csv`

In [ ]:
'''
Figure: crisis excess return over MCap monthly of the seven monthly LSTM
portfolios, for the second scatter plot in paper.tex (arrows from the
baseline to each regime variant).  Same computation as the frequency
scatter: cumulative return (metrics.cumulative_return) over (peak, trough]
and (trough, even], excess over MCap monthly as crisis_analysis_defined._excess,
mean over C1 ... C7, in %.
'''

import sys
from pathlib import Path

import numpy as np
import pandas as pd

HERE  = Path.cwd()                              # _latex, or the workspace root
PAPER = HERE.parent if (HERE.parent / '_comparison').is_dir() else HERE / 'Paper'
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics')]
import compare_all as ca
from crises import main_crises
from metrics import cumulative_return

FREQ       = 'monthly'
OUT_PATH_R = PAPER / '_latex' / 'data_finalized' / 'figure_scatter_lstm_regimes.csv'
BENCHMARK  = 'mcap_t_10'
# name in the figure -> (run, key for the style in paper.tex, side of the name
# label: north / south / east / west, chosen for the layout), in legend order
PORTFOLIOS = {
    'MCap':                    (BENCHMARK, 'mcap', 'west'),
    'LSTM Baseline':           ('lstm_t_10_holdout_rank', 'base', 'north'),
    'Changepoint Cash':        ('lstm_t_10_holdout_rank_changepoint_vix_cash', 'cpcash', 'east'),
    'Changepoint Alternative': ('lstm_t_10_holdout_rank_changepoint_vix_hrp', 'cpalt', 'east'),
    'Clustering Cash':         ('lstm_t_10_holdout_rank_wasserstein_gspc_vix_cash', 'clcash', 'east'),
    'Clustering Alternative':  ('lstm_t_10_holdout_rank_wasserstein_gspc_vix_hrp', 'clalt', 'east'),
    'HMM Cash':                ('lstm_t_10_holdout_rank_hmm_gspc_cash', 'hmmcash', 'east'),
    'HMM Alternative':         ('lstm_t_10_holdout_rank_hmm_gspc_hrp', 'hmmalt', 'east'),
}

by_run = {r['run']: r for r in ca.find_runs() if r['frequency'] == FREQ}

def phase_returns(run: str) -> pd.DataFrame:
    '''Cumulative return per crisis (rows C1 ... C7), columns ptt / ttp.'''
    if run not in by_run:
        raise KeyError(f'no {FREQ} run {run!r}')
    ret = by_run[run]['returns']
    return pd.DataFrame([{'ptt': cumulative_return(ca._slice(ret, c['peak'], c['trough'])),
                          'ttp': cumulative_return(ca._slice(ret, c['trough'], c['even']))}
                         for c in main_crises])

bench = phase_returns(BENCHMARK)

rows = []
for name, (run, key, pos) in PORTFOLIOS.items():
    e = 100 * ((1 + phase_returns(run)) / (1 + bench) - 1).mean()
    rows.append({'model': name, 'key': key, 'ptt': e['ptt'], 'ttp': e['ttp'], 'label_pos': pos})
figure_r = pd.DataFrame(rows)

base = figure_r.loc[figure_r['key'] == 'base', ['ptt', 'ttp']].iloc[0]
variant = ~figure_r['key'].isin(['mcap', 'base'])
figure_r['dx'] = np.where(variant, figure_r['ptt'] - base['ptt'], np.nan)
figure_r['dy'] = np.where(variant, figure_r['ttp'] - base['ttp'], np.nan)
figure_r['base_ptt'], figure_r['base_ttp'] = base['ptt'], base['ttp']   # arrow start
figure_r = figure_r[['model', 'key', 'ptt', 'ttp', 'base_ptt', 'base_ttp', 'dx', 'dy', 'label_pos']]

OUT_PATH_R.parent.mkdir(exist_ok=True)
figure_r.round(2).to_csv(OUT_PATH_R, index=False, na_rep='nan')
print(f'wrote {OUT_PATH_R}: {len(figure_r)} points')
figure_r.round(2)